In [1]:
import pandas as pd
import sys
import os

from nltk.corpus import stopwords
from sklearn.ensemble import AdaBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

if "Desktop" in os.getcwd():
    base_path = "/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification"
else:
    base_path = "/home/sakdag/weakly-supervised-fake-news-classification"

print(base_path)
sys.path.append(base_path)
    
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_raw_data
from src.preprocessing.read_raw_data import read_csv
from src.weak_labeling.consts.globals import NLP

/home/sakdag/weakly-supervised-fake-news-classification


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, train_test_split
from src.preprocessing.tfidf_vectorizer import load_tfidf_vectorizer
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from src.preprocessing.cleanup import clean_text
from sklearn import metrics

In [3]:
injected_df: pd.DataFrame = read_csv(Datasets.NELA_ELECTIONS_SUBSET_2020.injected_path)

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-elections-subset-2020-sampled.csv
Read dataset, dataframe size: 2000


In [4]:
from src.weak_labeling.model_based.adaboost.adaboost import (
    train_adaboost_model,
    get_adaboost_labels,
)

In [7]:
train_adaboost_model(
    dataset=Datasets.NELA_ELECTIONS_SUBSET_2020, save_identifier="jupyter-test"
)

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-elections-subset-2020-sampled.csv
Read dataset, dataframe size: 2000
accuracy: 0.642


In [8]:
train_adaboost_model(
    dataset=Datasets.NELA_COVID_2021, save_identifier="jupyter-test"
)

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-covid-2021-sampled.csv
Read dataset, dataframe size: 50000
accuracy: 0.731


In [13]:
injected_df: pd.DataFrame = read_csv(Datasets.NELA_COVID_2021.injected_path)

x = injected_df["content"]
y = injected_df["label"]
x_cleaned = x.apply(lambda t : clean_text(t))
X_train, X_test, y_train, y_test = train_test_split(
    x_cleaned, y, test_size=0.2, random_state=42
)

tfidf_vectorizer = load_tfidf_vectorizer("content")
train_vectors = tfidf_vectorizer.transform(X_train)
test_vectors = tfidf_vectorizer.transform(X_test)

params = {
    "base_estimator__max_depth": 10,
    "base_estimator__min_samples_leaf": 1,
    "learning_rate": 1.0,
    "n_estimators": 5,
}

abc = AdaBoostClassifier(
    DecisionTreeClassifier(
        max_depth=params["base_estimator__max_depth"],
        min_samples_leaf=params["base_estimator__min_samples_leaf"],
    ),
    n_estimators=params["n_estimators"],
    learning_rate=params["learning_rate"],
    random_state=1,
)
abc.fit(train_vectors, y_train)
y_pred = abc.predict(test_vectors)
abc_score = metrics.accuracy_score(y_test, y_pred)
print("accuracy: %0.3f" % abc_score)

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-covid-2021-sampled.csv
Read dataset, dataframe size: 50000
Loading TFIDF vectorizer from path: /home/sakdag/weakly-supervised-fake-news-classification/model/weak_labeling/tfidf_vectorizer/generic-content.pkl
accuracy: 0.765


In [15]:
injected_df: pd.DataFrame = read_csv(Datasets.NELA_ELECTIONS_SUBSET_2020.injected_path)

x = injected_df["content"]
y = injected_df["label"]
x_cleaned = x.apply(lambda t : clean_text(t))
X_train, X_test, y_train, y_test = train_test_split(
    x_cleaned, y, test_size=0.2, random_state=42
)

tfidf_vectorizer = load_tfidf_vectorizer("content")
train_vectors = tfidf_vectorizer.transform(X_train)
test_vectors = tfidf_vectorizer.transform(X_test)

params = {
    "base_estimator__max_depth": 10,
    "base_estimator__min_samples_leaf": 1,
    "learning_rate": 1.0,
    "n_estimators": 5,
}

abc = AdaBoostClassifier(
    DecisionTreeClassifier(
        max_depth=params["base_estimator__max_depth"],
        min_samples_leaf=params["base_estimator__min_samples_leaf"],
    ),
    n_estimators=params["n_estimators"],
    learning_rate=params["learning_rate"],
    random_state=1,
)
abc.fit(train_vectors, y_train)
y_pred = abc.predict(test_vectors)
abc_score = metrics.accuracy_score(y_test, y_pred)
print("accuracy: %0.3f" % abc_score)

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-elections-subset-2020-sampled.csv
Read dataset, dataframe size: 2000
Loading TFIDF vectorizer from path: /home/sakdag/weakly-supervised-fake-news-classification/model/weak_labeling/tfidf_vectorizer/generic-content.pkl
accuracy: 0.605


In [8]:
print(grid_search.best_params_)
print(grid_search.best_estimator_.base_estimator_.max_depth)

{'base_estimator__max_depth': 2, 'base_estimator__min_samples_leaf': 5, 'learning_rate': 0.1, 'n_estimators': 50}
2


In [ ]:
Adab = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=10), n_estimators=5, random_state=1
)